# Programación dinámica de intervalos

Muchos problemas sobre una secuencia se resuelven calculando la respuesta $D[i][j]$ de cada tramo $a_i \dots a_j$ a partir de tramos más cortos. Lo habitual es elegir un punto $k$ dentro del tramo (el último globo que se revienta, la última mezcla, el vértice que cierra un triángulo, la raíz de un árbol) y combinar las dos partes:

$$D[i][j] = \operatorname{opt}_{i \le k < j} \big( D[i][k] + D[k+1][j] + \text{coste}(i, k, j) \big)$$

La tabla se rellena por **longitud creciente** del tramo y cuesta $O(n^3)$ en tiempo y $O(n^2)$ en memoria; con la optimización de Knuth, cuando se cumple, baja a $O(n^2)$. *Subsecuencia palindrómica más larga* y *Multiplicación de matrices encadenada* son de esta familia; aquí van otros problemas clásicos, comprobados contra fuerza bruta.

## 1. Reventar globos

En una fila de globos con números $a_1, \dots, a_n$, reventar el globo $k$ da $a_{\text{izq}} \cdot a_k \cdot a_{\text{der}}$ puntos, donde izq y der son sus vecinos vivos en ese momento (fuera de la fila se cuenta un 1). Maximizar los puntos. La clave es pensar en el **último** globo $k$ que se revienta del tramo abierto $(i, j)$: cuando le toca, sus vecinos son $i$ y $j$, y los dos lados ya no se influyen:

$$D[i][j] = \max_{i < k < j} \big( D[i][k] + D[k][j] + a_i\, a_k\, a_j \big)$$

Reconstruir el orden en que hay que reventarlos y comprobarlo contra todas las permutaciones para $n \le 7$.

In [1]:
import random
from functools import cache
from itertools import accumulate, permutations


def globos(valores):
    a = [1, *valores, 1]
    n = len(a)
    D = [[0] * n for _ in range(n)]
    ultimo = [[None] * n for _ in range(n)]
    for longitud in range(2, n):
        for i in range(n - longitud):
            j = i + longitud
            for k in range(i + 1, j):
                puntos = D[i][k] + D[k][j] + a[i] * a[k] * a[j]
                if ultimo[i][j] is None or puntos > D[i][j]:
                    D[i][j], ultimo[i][j] = puntos, k

    def orden(i, j):
        k = ultimo[i][j]
        return [] if k is None else orden(i, k) + orden(k, j) + [k - 1]

    return D[0][n - 1], orden(0, n - 1)


def jugar(valores, orden):
    vivos, puntos = list(range(len(valores))), 0
    for g in orden:
        pos = vivos.index(g)
        izq = valores[vivos[pos - 1]] if pos > 0 else 1
        der = valores[vivos[pos + 1]] if pos + 1 < len(vivos) else 1
        puntos += izq * valores[g] * der
        vivos.pop(pos)
    return puntos


valores = [3, 1, 5, 8]
puntos, orden = globos(valores)
print(f"Globos {valores}: {puntos} puntos reventando las posiciones {orden}")
assert puntos == jugar(valores, orden) == 167

rng = random.Random(1)
for _ in range(150):
    valores = [rng.randint(0, 9) for _ in range(rng.randint(0, 7))]
    puntos, orden = globos(valores)
    assert puntos == jugar(valores, orden) == max(jugar(valores, p) for p in permutations(range(len(valores))))

Globos [3, 1, 5, 8]: 167 puntos reventando las posiciones [1, 2, 0, 3]


## 2. Partir una cadena en palíndromos con el mínimo de cortes

Partir una cadena en trozos que sean todos palíndromos con el menor número de cortes. Primero se marca en $P[i][j]$ qué tramos son palíndromos (un tramo lo es si sus extremos coinciden y el interior también lo es, por longitud creciente) y después $C_j$, el mínimo de cortes del prefijo $s_0 \dots s_j$, es 0 si el prefijo es palíndromo y si no $\min_{i \le j,\ P[i][j]} (C_{i-1} + 1)$. Todo cuesta $O(n^2)$. Mostrar la partición y comprobarlo probando las $2^{n-1}$ formas de cortar.

In [2]:
def palindromos_tabla(s):
    n = len(s)
    P = [[False] * n for _ in range(n)]
    for longitud in range(1, n + 1):
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            P[i][j] = s[i] == s[j] and (longitud <= 2 or P[i + 1][j - 1])
    return P


def cortes_minimos(s):
    if not s:
        return 0, []
    n = len(s)
    P = palindromos_tabla(s)
    C, desde = [0] * n, [0] * n
    for j in range(n):
        C[j] = n
        for i in range(j + 1):
            if P[i][j]:
                cortes = 0 if i == 0 else C[i - 1] + 1
                if cortes < C[j]:
                    C[j], desde[j] = cortes, i
    trozos, j = [], n - 1
    while j >= 0:
        trozos.append(s[desde[j]:j + 1])
        j = desde[j] - 1
    return C[n - 1], trozos[::-1]


def cortes_bruto(s):
    n = len(s)
    mejor = n
    for mascara in range(1 << max(n - 1, 0)):
        bordes = [0] + [i + 1 for i in range(n - 1) if mascara >> i & 1] + [n]
        trozos = [s[a:b] for a, b in zip(bordes, bordes[1:])]
        if all(t == t[::-1] for t in trozos):
            mejor = min(mejor, len(trozos) - 1)
    return mejor


for palabra in ["anilina", "arrozalazorra", "radarsolosananas", "abcbadd", "tenerife"]:
    c, trozos = cortes_minimos(palabra)
    print(f"{palabra:>18}: {c} cortes  {' | '.join(trozos)}")
    assert c == cortes_bruto(palabra) == len(trozos) - 1 and "".join(trozos) == palabra

rng = random.Random(2)
for _ in range(300):
    s = "".join(rng.choices("ab" if rng.random() < 0.5 else "abc", k=rng.randint(1, 13)))
    c, trozos = cortes_minimos(s)
    assert c == cortes_bruto(s) == len(trozos) - 1
    assert "".join(trozos) == s and all(t == t[::-1] for t in trozos)

           anilina: 0 cortes  anilina
     arrozalazorra: 0 cortes  arrozalazorra
  radarsolosananas: 3 cortes  radar | s | olo | sananas
           abcbadd: 1 cortes  abcba | dd
          tenerife: 5 cortes  t | ene | r | i | f | e


## 3. Mezclar montones de piedras

En un muelle hay sacos de arena en fila formando montones de pesos $a_1, \dots, a_n$. Solo se pueden juntar dos montones vecinos y cuesta lo que pesan los dos juntos. Calcular el coste mínimo de juntarlo todo: la última mezcla une $a_i \dots a_k$ con $a_{k+1} \dots a_j$ y cuesta el peso del tramo, que con sumas acumuladas es $S_{j+1} - S_i$:

$$D[i][j] = \min_{i \le k < j} \big( D[i][k] + D[k+1][j] \big) + (S_{j+1} - S_i)$$

Escribir el orden de las mezclas con paréntesis y comprobarlo contra una búsqueda que prueba todas las secuencias de mezclas posibles.

In [3]:
def piedras(a):
    n = len(a)
    S = [0, *accumulate(a)]
    D = [[0] * n for _ in range(n)]
    corte = [[0] * n for _ in range(n)]
    for longitud in range(2, n + 1):
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            D[i][j], corte[i][j] = min((D[i][k] + D[k + 1][j], k) for k in range(i, j))
            D[i][j] += S[j + 1] - S[i]

    def escribe(i, j):
        if i == j:
            return str(a[i])
        k = corte[i][j]
        return f"({escribe(i, k)} + {escribe(k + 1, j)})"

    return D[0][n - 1], escribe(0, n - 1), D, corte


@cache
def piedras_bruto(montones):
    if len(montones) == 1:
        return 0
    return min(
        montones[i] + montones[i + 1] + piedras_bruto(montones[:i] + (montones[i] + montones[i + 1],) + montones[i + 2:])
        for i in range(len(montones) - 1)
    )


montones = [4, 1, 1, 4]
coste, mezclas, *_ = piedras(montones)
print(f"{montones}: coste {coste} con {mezclas}")
montones = [6, 4, 4, 6]
coste, mezclas, *_ = piedras(montones)
print(f"{montones}: coste {coste} con {mezclas}")
assert piedras([4, 1, 1, 4])[0] == 18 and coste == 40

rng = random.Random(3)
for _ in range(200):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]
    assert piedras(a)[0] == piedras_bruto(tuple(a))

[4, 1, 1, 4]: coste 18 con (4 + ((1 + 1) + 4))
[6, 4, 4, 6]: coste 40 con ((6 + 4) + (4 + 6))


## 4. Optimización de Knuth

En el problema de las piedras, el mejor punto de corte es monótono: $K[i][j-1] \le K[i][j] \le K[i+1][j]$ (el coste del tramo cumple la desigualdad del cuadrángulo). Basta entonces probar $k$ en ese rango y la suma de todos los rangos de una misma longitud se telescopa, así que el total baja a $O(n^2)$. Implementarlo contando cuántos $k$ se prueban, comprobar que da lo mismo que la versión $O(n^3)$ en listas aleatorias y comparar el número de pruebas.

In [4]:
def piedras_knuth(a):
    n = len(a)
    S = [0, *accumulate(a)]
    D = [[0] * n for _ in range(n)]
    K = [[i] * n for i in range(n)]
    pruebas = 0
    for longitud in range(2, n + 1):
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            mejor = None
            for k in range(K[i][j - 1], min(K[i + 1][j], j - 1) + 1):
                pruebas += 1
                valor = D[i][k] + D[k + 1][j]
                if mejor is None or valor < mejor:
                    mejor, K[i][j] = valor, k
            D[i][j] = mejor + S[j + 1] - S[i]
    return (D[0][n - 1] if n else 0), pruebas


rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(1, 100) for _ in range(rng.randint(1, 40))]
    assert piedras_knuth(a)[0] == piedras(a)[0]

print(f"{'n':>4} {'pruebas O(n³)':>14} {'pruebas Knuth':>14}")
for n in (10, 50, 100, 200):
    a = [rng.randint(1, 100) for _ in range(n)]
    coste, pruebas = piedras_knuth(a)
    cubica = sum((longitud - 1) * (n - longitud + 1) for longitud in range(2, n + 1))
    print(f"{n:>4} {cubica:>14} {pruebas:>14}")
    assert pruebas < 2 * n * n

   n  pruebas O(n³)  pruebas Knuth
  10            165             86
  50          20825           2516
 100         166650           9750
 200        1333300          39821


## 5. Montones en círculo

Si los montones están alrededor de una plaza en círculo (el último es vecino del primero), la última mezcla une dos arcos. Duplicar la lista, $a_1, \dots, a_n, a_1, \dots, a_n$, rellenar la tabla para tramos de hasta $n$ montones y quedarse con el mejor tramo de longitud $n$. Comprobarlo contra fuerza bruta con mezclas circulares y comprobar que nunca cuesta más que en fila.

In [5]:
def piedras_circulo(a):
    n = len(a)
    if n == 1:
        return 0
    doble = a + a
    S = [0, *accumulate(doble)]
    D = [[0] * (2 * n) for _ in range(2 * n)]
    for longitud in range(2, n + 1):
        for i in range(2 * n - longitud + 1):
            j = i + longitud - 1
            D[i][j] = min(D[i][k] + D[k + 1][j] for k in range(i, j)) + S[j + 1] - S[i]
    return min(D[i][i + n - 1] for i in range(n))


@cache
def circulo_bruto(montones):
    n = len(montones)
    if n == 1:
        return 0
    mejor = float("inf")
    for i in range(n):
        j = (i + 1) % n
        juntos = montones[i] + montones[j]
        resto = [m for t, m in enumerate(montones) if t not in (i, j)]
        nuevo = (juntos, *resto[i:], *resto[:i]) if j else (juntos, *resto)
        mejor = min(mejor, juntos + circulo_bruto(nuevo))
    return mejor


plaza = [1, 9, 9, 1]
print(f"{plaza}: en fila {piedras(plaza)[0]}, en círculo {piedras_circulo(plaza)}")
rng = random.Random(5)
for _ in range(200):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 7))]
    assert piedras_circulo(a) == circulo_bruto(tuple(a)) <= piedras(a)[0]

[1, 9, 9, 1]: en fila 40, en círculo 33


## 6. Triangulación de un polígono de coste mínimo

Los vértices de un polígono convexo tienen pesos $w_0, \dots, w_{n-1}$ y un triángulo $(i, k, j)$ cuesta $w_i\, w_k\, w_j$. Partir el polígono en $n - 2$ triángulos con diagonales que no se crucen y con el menor coste total. El lado $(i, j)$ pertenece a un único triángulo $(i, k, j)$, que separa el polígono en dos más pequeños:

$$D[i][j] = \min_{i < k < j} \big( D[i][k] + D[k][j] + w_i\, w_k\, w_j \big)$$

Listar los triángulos de la solución y comprobarlo enumerando todas las triangulaciones (hay un número de Catalan $C_{n-2}$ de ellas). Repetirlo con un coste geométrico: los vértices de un heptágono irregular y el perímetro de cada triángulo.

In [6]:
from math import comb, dist


def triangulacion(n, coste):
    D = [[0.0] * n for _ in range(n)]
    elegido = [[None] * n for _ in range(n)]
    for longitud in range(2, n):
        for i in range(n - longitud):
            j = i + longitud
            D[i][j], elegido[i][j] = min((D[i][k] + D[k][j] + coste(i, k, j), k) for k in range(i + 1, j))

    def triangulos(i, j):
        k = elegido[i][j]
        return [] if k is None else triangulos(i, k) + [(i, k, j)] + triangulos(k, j)

    return D[0][n - 1], triangulos(0, n - 1)


def todas_triangulaciones(i, j):
    if j - i < 2:
        return [[]]
    return [izq + [(i, k, j)] + der for k in range(i + 1, j)
            for izq in todas_triangulaciones(i, k) for der in todas_triangulaciones(k, j)]


pesos = [3, 7, 4, 5, 1, 6]
coste_pesos = lambda i, k, j: pesos[i] * pesos[k] * pesos[j]
valor, triangulos = triangulacion(len(pesos), coste_pesos)
print(f"Pesos {pesos}: coste {valor:.0f} con los triángulos {triangulos}")
todas = todas_triangulaciones(0, len(pesos) - 1)
assert len(todas) == comb(2 * 4, 4) // 5
assert valor == min(sum(coste_pesos(*t) for t in tri) for tri in todas)

rng = random.Random(6)
for _ in range(60):
    n = rng.randint(3, 9)
    w = [rng.randint(1, 9) for _ in range(n)]
    c = lambda i, k, j: w[i] * w[k] * w[j]
    valor, tri = triangulacion(n, c)
    assert len(tri) == n - 2 and valor == sum(c(*t) for t in tri)
    assert valor == min(sum(c(*t) for t in t_) for t_ in todas_triangulaciones(0, n - 1))

heptagono = [(0, 0), (4, -1), (7, 1), (8, 4), (6, 7), (2, 7), (-1, 4)]
perimetro = lambda i, k, j: dist(heptagono[i], heptagono[k]) + dist(heptagono[k], heptagono[j]) + dist(heptagono[i], heptagono[j])
valor, tri = triangulacion(len(heptagono), perimetro)
print(f"Heptágono: suma de perímetros {valor:.3f} con {tri}")
assert abs(valor - min(sum(perimetro(*t) for t in t_) for t_ in todas_triangulaciones(0, 6))) < 1e-9

Pesos [3, 7, 4, 5, 1, 6]: coste 87 con los triángulos [(0, 1, 4), (1, 2, 4), (2, 3, 4), (0, 4, 5)]
Heptágono: suma de perímetros 83.351 con [(0, 1, 2), (0, 2, 5), (2, 3, 4), (2, 4, 5), (0, 5, 6)]


## 7. Árbol binario de búsqueda óptimo

Un diccionario de guanchismos guarda sus palabras ordenadas en un árbol binario de búsqueda y se sabe cuántas veces se consulta cada una, $f_i$. Buscar una palabra a profundidad $p$ (la raíz es 1) cuesta $p$ comparaciones. Elegir el árbol que minimiza el total $\sum f_i\, p_i$: si la raíz del tramo es $r$, todas las palabras del tramo bajan un nivel, así que

$$D[i][j] = \min_{i \le r \le j} \big( D[i][r-1] + D[r+1][j] \big) + \sum_{t=i}^{j} f_t$$

Dibujar el árbol óptimo por niveles y comprobarlo construyendo todos los árboles posibles para $n \le 8$.

In [7]:
def abb_optimo(frec):
    n = len(frec)
    S = [0, *accumulate(frec)]

    @cache
    def D(i, j):
        if i > j:
            return 0, None
        return min((D(i, r - 1)[0] + D(r + 1, j)[0] + S[j + 1] - S[i], r) for r in range(i, j + 1))

    def arbol(i, j):
        if i > j:
            return None
        r = D(i, j)[1]
        return (r, arbol(i, r - 1), arbol(r + 1, j))

    return D(0, n - 1)[0], arbol(0, n - 1)


def todos_los_arboles(i, j):
    if i > j:
        return [None]
    return [(r, izq, der) for r in range(i, j + 1)
            for izq in todos_los_arboles(i, r - 1) for der in todos_los_arboles(r + 1, j)]


def coste_arbol(arbol, frec, profundidad=1):
    if arbol is None:
        return 0
    r, izq, der = arbol
    return frec[r] * profundidad + coste_arbol(izq, frec, profundidad + 1) + coste_arbol(der, frec, profundidad + 1)


def niveles(arbol, palabras):
    nivel, salida = [arbol], []
    while nivel:
        salida.append("  ".join(palabras[n[0]] for n in nivel))
        nivel = [h for n in nivel for h in n[1:] if h is not None]
    return salida


palabras = ["baifo", "beñesmén", "gofio", "goro", "guanil", "jaira", "tabaiba", "tafeña"]
consultas = [40, 5, 60, 8, 3, 25, 30, 4]
coste, arbol = abb_optimo(consultas)
print("Coste:", coste)
for linea in niveles(arbol, palabras):
    print("  ", linea)
assert coste == coste_arbol(arbol, consultas) == min(coste_arbol(t, consultas) for t in todos_los_arboles(0, 7))

rng = random.Random(7)
for _ in range(80):
    frec = [rng.randint(0, 20) for _ in range(rng.randint(1, 7))]
    coste, arbol = abb_optimo(frec)
    assert coste == coste_arbol(arbol, frec) == min(coste_arbol(t, frec) for t in todos_los_arboles(0, len(frec) - 1))

Coste: 347
   gofio
   baifo  jaira
   beñesmén  goro  tabaiba
   guanil  tafeña


## 8. El juego de las monedas en fila

Dos jugadoras se turnan para coger una moneda de uno de los dos extremos de una fila. Si las dos juegan perfecto, ¿por cuánto gana la primera? Con $D[i][j]$ la máxima ventaja (lo suyo menos lo de la rival) que consigue quien mueve en el tramo $i \dots j$:

$$D[i][j] = \max\big(a_i - D[i+1][j],\ a_j - D[i][j-1]\big)$$

Comprobarlo con un minimax sin memoria para filas cortas, enseñar una partida perfecta y comprobar que con un número par de monedas la primera nunca pierde (le basta con llevarse todas las de posición par o todas las de posición impar, la mejor de las dos).

In [8]:
def monedas(a):
    n = len(a)
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        D[i][i] = a[i]
    for longitud in range(2, n + 1):
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            D[i][j] = max(a[i] - D[i + 1][j], a[j] - D[i][j - 1])
    return D


def minimax(a):
    if not a:
        return 0
    return max(a[0] - minimax(a[1:]), a[-1] - minimax(a[:-1]))


fila = [5, 3, 7, 10, 2, 6]
D = monedas(fila)
i, j, turno, marcador = 0, len(fila) - 1, 0, [0, 0]
while i <= j:
    if i == j or fila[i] - D[i + 1][j] >= fila[j] - D[i][j - 1]:
        coge, i = fila[i], i + 1
    else:
        coge, j = fila[j], j - 1
    marcador[turno] += coge
    print(f"Jugadora {turno + 1} coge {coge:>2}  ->  {marcador}")
    turno = 1 - turno
print("Ventaja de la primera:", D[0][-1])
assert marcador[0] - marcador[1] == D[0][-1] == minimax(fila)

rng = random.Random(8)
for _ in range(200):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]
    D = monedas(a)
    assert D[0][-1] == minimax(a)
    if len(a) % 2 == 0:
        pares, impares = sum(a[0::2]), sum(a[1::2])
        assert D[0][-1] >= abs(pares - impares) >= 0

Jugadora 1 coge  6  ->  [6, 0]
Jugadora 2 coge  5  ->  [6, 5]
Jugadora 1 coge  3  ->  [9, 5]
Jugadora 2 coge  7  ->  [9, 12]
Jugadora 1 coge 10  ->  [19, 12]
Jugadora 2 coge  2  ->  [19, 14]
Ventaja de la primera: 5


## 9. El problema de los huevos: tabla clásica

En una torre de $n$ pisos hay un piso crítico desconocido $c \in \{0, \dots, n\}$: un huevo lanzado desde un piso $\le c$ no se rompe y desde uno superior sí. Con $k$ huevos, ¿cuántos lanzamientos hacen falta en el peor caso para encontrar $c$? Si se lanza desde el piso $x$ y se rompe, quedan $k - 1$ huevos y $x - 1$ pisos; si no, $k$ huevos y $n - x$ pisos:

$$W(k, n) = 1 + \min_{1 \le x \le n} \max\big( W(k-1, x-1),\ W(k, n-x) \big), \qquad W(1, n) = n, \quad W(k, 0) = 0$$

Rellenar la tabla en $O(k\, n^2)$ y comprobar que con 2 huevos y 100 pisos hacen falta 14, que con $k \ge \lceil \log_2(n+1) \rceil$ huevos basta la búsqueda binaria y que con 2 huevos el resultado es el menor $t$ con $t(t+1)/2 \ge n$.

In [9]:
from math import ceil, log2


def huevos_tabla(k_max, n_max):
    W = [[0] * (n_max + 1) for _ in range(k_max + 1)]
    for n in range(n_max + 1):
        W[1][n] = n
    for k in range(2, k_max + 1):
        for n in range(1, n_max + 1):
            W[k][n] = 1 + min(max(W[k - 1][x - 1], W[k][n - x]) for x in range(1, n + 1))
    return W


W = huevos_tabla(5, 120)
print(f"{'pisos':>6}" + "".join(f"{f'k = {k}':>8}" for k in range(1, 6)))
for n in (1, 2, 10, 36, 100, 120):
    print(f"{n:>6}" + "".join(f"{W[k][n]:>8}" for k in range(1, 6)))
assert W[2][100] == 14
for n in range(1, 121):
    assert W[2][n] == next(t for t in range(n + 1) if t * (t + 1) // 2 >= n)
    k = ceil(log2(n + 1))
    if k <= 5:
        assert W[k][n] == k

 pisos   k = 1   k = 2   k = 3   k = 4   k = 5
     1       1       1       1       1       1
     2       2       2       2       2       2
    10      10       4       4       4       4
    36      36       8       6       6       6
   100     100      14       9       8       7
   120     120      15       9       8       8


## 10. El problema de los huevos por número de intentos

Dándole la vuelta a la pregunta, sea $F(t, k)$ el máximo de pisos que se pueden resolver con $t$ lanzamientos y $k$ huevos. El primer lanzamiento se hace desde el piso $F(t-1, k-1) + 1$: si se rompe, los de abajo se resuelven con lo que queda; si no, encima caben $F(t-1, k)$ pisos más:

$$F(t, k) = F(t-1, k-1) + F(t-1, k) + 1 = \sum_{i=1}^{k} \binom{t}{i}$$

La respuesta es el menor $t$ con $F(t, k) \ge n$, que se encuentra por búsqueda binaria sobre $t$ en $O(k \log n)$. Comprobar contra la tabla del ejercicio anterior y resolver torres enormes (por ejemplo, $10^{18}$ pisos).

In [10]:
def pisos_resolubles(t, k, tope):
    """F(t, k) con la suma de binomiales, cortando en `tope` para no hacer cuentas gigantes."""
    total, binomial = 0, 1
    for i in range(1, k + 1):
        binomial = binomial * (t - i + 1) // i
        total += binomial
        if total >= tope:
            return tope
    return total


def huevos(k, n):
    lo, hi = 0, n
    while lo < hi:
        t = (lo + hi) // 2
        if pisos_resolubles(t, k, n) >= n:
            hi = t
        else:
            lo = t + 1
    return lo


@cache
def F(t, k):
    return 0 if t == 0 or k == 0 else F(t - 1, k - 1) + F(t - 1, k) + 1


for t in range(30):
    for k in range(1, 8):
        assert F(t, k) == sum(comb(t, i) for i in range(1, k + 1)) == pisos_resolubles(t, k, 10**30)
for k in range(1, 6):
    for n in range(121):
        assert huevos(k, n) == W[k][n]

print(f"{'huevos':>6} {'10^6 pisos':>11} {'10^18 pisos':>12}")
for k in (1, 2, 3, 5, 10, 64):
    print(f"{k:>6} {huevos(k, 10**6):>11} {huevos(k, 10**18):>12}")
assert huevos(2, 10**6) == 1414 and huevos(64, 10**18) == 60

huevos  10^6 pisos  10^18 pisos
     1     1000000 1000000000000000000
     2        1414   1414213562
     3         182      1817121
     5          43        10373
    10          21          290
    64          20           60


## 11. Jugar la estrategia óptima

Convertir la fórmula en una estrategia: con $t$ lanzamientos y $k$ huevos disponibles y la zona de búsqueda empezando en el piso $b$, lanzar desde $b + F(t-1, k-1) + 1$ (o desde el último piso de la zona, si es más bajo). Simularla contra todos los pisos críticos posibles para varias torres, comprobando que siempre acierta, que nunca rompe más huevos de los que hay y que el peor caso usa exactamente `huevos(k, n)` lanzamientos. Mostrar la partida con 2 huevos, 100 pisos y piso crítico 73.

In [11]:
def jugar_huevos(k, n, critico, traza=False):
    t = huevos(k, n)
    abajo, arriba = 0, n  # el crítico está en [abajo, arriba]
    lanzamientos = 0
    while abajo < arriba:
        piso = min(abajo + F(t - 1, k - 1) + 1, arriba)
        lanzamientos += 1
        t -= 1
        if piso > critico:
            k -= 1
            arriba = piso - 1
            resultado = "se rompe"
        else:
            abajo = piso
            resultado = "aguanta"
        if traza:
            print(f"  lanzamiento {lanzamientos:>2}: piso {piso:>3} {resultado:<9} crítico en [{abajo}, {arriba}]")
        assert k >= 0 and t >= 0
    return abajo, lanzamientos


print("2 huevos, 100 pisos, crítico 73:")
encontrado, usados = jugar_huevos(2, 100, 73, traza=True)
assert encontrado == 73

for k in range(1, 5):
    for n in range(0, 80):
        peor = 0
        for critico in range(n + 1):
            encontrado, usados = jugar_huevos(k, n, critico)
            assert encontrado == critico
            peor = max(peor, usados)
        assert peor == huevos(k, n)

2 huevos, 100 pisos, crítico 73:
  lanzamiento  1: piso  14 aguanta   crítico en [14, 100]
  lanzamiento  2: piso  27 aguanta   crítico en [27, 100]
  lanzamiento  3: piso  39 aguanta   crítico en [39, 100]
  lanzamiento  4: piso  50 aguanta   crítico en [50, 100]
  lanzamiento  5: piso  60 aguanta   crítico en [60, 100]
  lanzamiento  6: piso  69 aguanta   crítico en [69, 100]
  lanzamiento  7: piso  77 se rompe  crítico en [69, 76]
  lanzamiento  8: piso  70 aguanta   crítico en [70, 76]
  lanzamiento  9: piso  71 aguanta   crítico en [71, 76]
  lanzamiento 10: piso  72 aguanta   crítico en [72, 76]
  lanzamiento 11: piso  73 aguanta   crítico en [73, 76]
  lanzamiento 12: piso  74 se rompe  crítico en [73, 73]
